## MODEL TRAINING
- logistic regression
- random forest

- Split the data into 80% training and 20% testing so we have unseen data to evaluate on
- Applied SMOTE on the training data only and not on the test data, to avoid data leakage
- Scaled the features using StandardScaler so all numbers are on the same scale
- Trained three models and compared their performance:
   - Logistic Regression gave 0.9851 translating to 93% correct ranking
   - Random Forest gave 0.9940 translating to 92% correct ranking
   - XGBoost gave  0.9962 translating to 95% correct ranking

XGBoost correctly ranks a defaulter as higher risk than a non-defaulter 99.62% of the time. It also caught 95% of all actual 
defaulters in the test set, which is an important metric for a bank missing a defaulter is more costly than a false alarm.The model (XGBoost) and the scaler were saved to the `models/` folder so they can 
be loaded later for deployment without retraining . 

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import joblib
import warnings
warnings.filterwarnings('ignore')

In [2]:
train = pd.read_csv(r'C:\Users\User\Desktop\credit-risk-project\data\processed\train_features.csv')

X = train.drop(columns=['target'])
y = train['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Default rate:", y_train.mean().round(4))

Train: (54923, 34) Test: (13731, 34)
Default rate: 0.0183


In [ ]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print("After SMOTE:", X_train_sm.shape)
print("Default rate after SMOTE:", y_train_sm.mean().round(4)) #50%

After SMOTE: (107834, 34)
Default rate after SMOTE: 0.5


- **SMOTE (Synthetic Minority Oversampling Technique)**, 
artificially generates new examples of the minority class (defaulters) so the model 
gets enough examples to learn from. After SMOTE, the training data was balanced 50/50.


In [4]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_sm)
X_test_sc = scaler.transform(X_test)

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train_sm)
lr_preds = lr.predict(X_test_sc)
lr_proba = lr.predict_proba(X_test_sc)[:,1]
print("Logistic Regression")
print(classification_report(y_test, lr_preds))
print("ROC-AUC:", round(roc_auc_score(y_test, lr_proba), 4))

=== Logistic Regression ===
              precision    recall  f1-score   support

           0       1.00      0.96      0.98     13479
           1       0.33      0.93      0.48       252

    accuracy                           0.96     13731
   macro avg       0.66      0.95      0.73     13731
weighted avg       0.99      0.96      0.97     13731

ROC-AUC: 0.9851


In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)
rf.fit(X_train_sm, y_train_sm)
rf_preds = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:,1]
print("Random Forest ")
print(classification_report(y_test, rf_preds))
print("ROC-AUC:", round(roc_auc_score(y_test, rf_proba), 4))

=== Random Forest ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     13479
           1       0.78      0.92      0.85       252

    accuracy                           0.99     13731
   macro avg       0.89      0.96      0.92     13731
weighted avg       0.99      0.99      0.99     13731

ROC-AUC: 0.994


In [ ]:
xgb = XGBClassifier(n_estimators=100, random_state=42, scale_pos_weight=54, eval_metric='logloss', n_jobs=-1)
xgb.fit(X_train_sm, y_train_sm)
xgb_preds = xgb.predict(X_test)
xgb_proba = xgb.predict_proba(X_test)[:,1]
print("XGBoost")
print(classification_report(y_test, xgb_preds))
print("ROC-AUC:", round(roc_auc_score(y_test, xgb_proba), 4))

=== XGBoost ===
              precision    recall  f1-score   support

           0       1.00      0.99      1.00     13479
           1       0.71      0.95      0.81       252

    accuracy                           0.99     13731
   macro avg       0.86      0.97      0.90     13731
weighted avg       0.99      0.99      0.99     13731

ROC-AUC: 0.9962


In [9]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'ROC-AUC': [
        round(roc_auc_score(y_test, lr_proba), 4),
        round(roc_auc_score(y_test, rf_proba), 4),
        round(roc_auc_score(y_test, xgb_proba), 4)
    ]
})
print(results.sort_values('ROC-AUC', ascending=False))

                 Model  ROC-AUC
2              XGBoost   0.9962
1        Random Forest   0.9940
0  Logistic Regression   0.9851


In [10]:
joblib.dump(xgb, r'C:\Users\User\Desktop\credit-risk-project\models\best_model.pkl')
joblib.dump(scaler, r'C:\Users\User\Desktop\credit-risk-project\models\pipeline.pkl')
joblib.dump(X.columns.tolist(), r'C:\Users\User\Desktop\credit-risk-project\models\feature_names.pkl')
print("Models saved")

Models saved
